In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import lightgbm as lgb
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


# 1. Data Loading 

In [2]:
data_path = 'data/'
try:
    train_df = pd.read_csv(data_path + 'train.csv')
    test_df = pd.read_csv(data_path + 'test.csv')
    sample_submission_df = pd.read_csv(data_path + 'sample_submission.csv')
except FileNotFoundError as e:
    train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
    test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')
    sample_submission_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv')

print("Train data shape:", train_df.shape)
print("Test data shape:", test_df.shape)

# 2. Data Cleaning & Quality Check

In this section, we perform a basic data quality check by identifying missing values, duplicates, and examining the distribution of features to ensure there are no unexpected outliers or data entry errors.

In [3]:
print(f"Total missing values: {train_df.isnull().sum().sum()}")
print(f"Total duplicate rows: {train_df.duplicated().sum()}")
train_df.describe()

# 3. Feature Engineering

Feature engineering is a critical step in building robust machine learning models. In this section, we implement advanced techniques to extract maximum signal from the data while avoiding redundant or collinear features:

1. **Frequency Encoding**: We encode categorical variables (`Crop_Type`, `Soil_Type`, `Region`, `Season`) by their frequency. This helps the model identify common vs. rare environmental configurations which often correlate with standard irrigation practices.
2. **Ordinal Target Encoding**: We map the target `Irrigation_Need` to numerical values (Low=0, Medium=1, High=2) and calculate the mean for each crop and soil type. This provides a direct supervised signal that quantifies the 'thirst' of different crops.
3. **Quantile Binning**: Continuous variables like `Rainfall_mm` are discretized into 10 bins. This allows the tree-based model to capture non-linear 'step' effects in water requirements more easily.
4. **Non-linear Proxies**: We create interaction features like `Water_Stress_Index` (Evapotranspiration proxy) and `Rain_Moisture_Ratio` to capture complex physiological relationships that a model might miss with raw features alone.

In [ ]:
def create_features(train, test):
    # Combine for global frequency stats
    df_all = pd.concat([train, test], axis=0).reset_index(drop=True)
    
    # 1. Frequency Encoding
    for col in ['Crop_Type', 'Soil_Type', 'Region', 'Season']:
        freq_map = df_all[col].value_counts().to_dict()
        train[f'{col}_Freq'] = train[col].map(freq_map)
        test[f'{col}_Freq'] = test[col].map(freq_map)

    # 2. Target Encoding (using ordinal mapping)
    target_map = {'Low': 0, 'Medium': 1, 'High': 2}
    train['target_num'] = train['Irrigation_Need'].map(target_map)
    
    for col in ['Crop_Type', 'Soil_Type']:
        target_mean = train.groupby(col)['target_num'].mean().to_dict()
        train[f'{col}_Target_Mean'] = train[col].map(target_mean)
        test[f'{col}_Target_Mean'] = test[col].map(target_mean)
    
    train.drop('target_num', axis=1, inplace=True)

    # 3. Binning Top Features
    train['Rainfall_Bin'] = pd.qcut(train['Rainfall_mm'], 10, labels=False)
    test['Rainfall_Bin'] = pd.qcut(test['Rainfall_mm'], 10, labels=False)
    
    # 4. High-impact Non-linear proxies
    for df in [train, test]:
        df['Water_Stress_Index'] = (df['Temperature_C'] * df['Wind_Speed_kmh']) / (df['Humidity'] + 1)
        df['Rain_Moisture_Ratio'] = df['Rainfall_mm'] / (df['Soil_Moisture'] + 1)

    return train, test

train_df, test_df = create_features(train_df, test_df)
print("Advanced features added. Shape:", train_df.shape)

### 3.1 Feature Correlation Analysis

To ensure our new features are not redundant or adding noise, we'll examine their correlation with the target and their base components. High correlation between features (multicollinearity) can sometimes negatively impact model performance.

In [ ]:
def analyze_correlation(df, target_col):
    temp_df = df.copy()
    le = LabelEncoder()
    if temp_df[target_col].dtype == 'object':
        temp_df[target_col] = le.fit_transform(temp_df[target_col])
    corr_matrix = temp_df.select_dtypes(include=[np.number]).corr()
    plt.figure(figsize=(12, 8))
    target_corr = corr_matrix[target_col].sort_values(ascending=False)
    sns.barplot(x=target_corr.index, y=target_corr.values)
    plt.title(f'Feature Correlation with {target_col}')
    plt.xticks(rotation=90)
    plt.show()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
    print("Features with > 0.9 correlation with another feature:")
    print(to_drop)
    return target_corr

target_correlations = analyze_correlation(train_df, 'Irrigation_Need')

# 4. Data Preprocessing

In [ ]:
X = train_df.drop('Irrigation_Need', axis=1)
y = train_df['Irrigation_Need']

# Re-map target to 0, 1, 2 for consistency across models
target_map = {'Low': 0, 'Medium': 1, 'High': 2}
y = y.map(target_map)

categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(include=['number']).columns

for col in categorical_features:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    test_df[col] = le.transform(test_df[col])

scaler = StandardScaler()
numerical_features_no_id = [col for col in numerical_features if col != 'id']
X[numerical_features_no_id] = scaler.fit_transform(X[numerical_features_no_id])
test_df[numerical_features_no_id] = scaler.transform(test_df[numerical_features_no_id])

# 5. Model Training and Ensembling

In this section, we train multiple high-performance gradient boosting models and combine them into an ensemble. Ensembling works by averaging the strengths of different models while minimizing their individual weaknesses.

1. **LightGBM**: Trains using a leaf-wise growth strategy, known for speed and accuracy.
2. **XGBoost**: Trains using a level-wise growth strategy, providing a complementary perspective to LightGBM.
3. **Weighted Ensemble**: We combine the predicted probabilities of both models. This often results in a more stable and accurate prediction than any single model.

In [ ]:
X_sample, _, y_sample, _ = train_test_split(X, y, train_size=0.1, stratify=y, random_state=42)
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'num_leaves': [31, 50],
}
lgbm = lgb.LGBMClassifier(random_state=42, class_weight='balanced', force_row_wise=True)
grid_search = GridSearchCV(estimator=lgbm, param_grid=param_grid, cv=3, scoring='f1_weighted', n_jobs=-1, verbose=1)
grid_search.fit(X_sample, y_sample)
best_params_lgb = grid_search.best_params_
print("Best parameters for LightGBM: ", best_params_lgb)

In [ ]:
print("Training LightGBM model...")
model_lgb = lgb.LGBMClassifier(random_state=42, class_weight='balanced', force_row_wise=True, **best_params_lgb)
model_lgb.fit(X, y)

print("Training XGBoost model...")
model_xgb = XGBClassifier(random_state=42, n_estimators=200, learning_rate=0.1, max_depth=6, tree_method='hist')
model_xgb.fit(X, y)

## 5.1 Feature Importance Analysis

In [ ]:
feature_imp = pd.DataFrame({'Value': model_lgb.feature_importances_, 'Feature': X.columns})
plt.figure(figsize=(10, 8))
sns.barplot(x="Value", y="Feature", data=feature_imp.sort_values(by="Value", ascending=False).head(20))
plt.title('LightGBM Top Features (Importance)')
plt.tight_layout()
plt.show()

# 6. Prediction and Submission

We use a **Soft Voting** approach, where we average the predicted probabilities from LightGBM and XGBoost. This ensures that the final prediction is based on the consensus of both models.

In [ ]:
print("Generating Ensemble Predictions...")
lgb_probs = model_lgb.predict_proba(test_df)
xgb_probs = model_xgb.predict_proba(test_df)

# Simple average of probabilities (0.5 weight each)
ensemble_probs = (lgb_probs + xgb_probs) / 2
ensemble_preds = np.argmax(ensemble_probs, axis=1)

# Map back to original labels
reverse_map = {0: 'Low', 1: 'Medium', 2: 'High'}
final_preds = [reverse_map[p] for p in ensemble_preds]

submission_df = pd.DataFrame({'id': test_df['id'], 'Irrigation_Need': final_preds})
submission_df.to_csv('submission.csv', index=False)
print("Ensemble submission file created successfully!")